# 04 - Kampanye tuning hyperparameter

Menjalankan grid yang dirancang di `tuning_grids/`. Prinsipnya human-in-the-loop:
tidak ada pencarian otomatis, urutan konfigurasi ditentukan manusia, dan setiap
baris hasil membawa kolom `catatan` yang merekam alasan konfigurasi itu dicoba.

Seleksi memakai split validation. Split test tidak disentuh sama sekali di
notebook ini.

Metode: grid kombinatorial untuk sumbu yang saling terkait, coordinate descent
untuk sumbu yang independen. Untuk RM-a, `lr`, `epochs`, dan `batch` bersama-sama
menentukan lintasan optimasi (batch 32 pada 5 epoch memberi separuh jumlah
langkah pembaruan dibanding batch 16), sehingga ketiganya harus digrid bersama;
`warmup_ratio` dan `weight_decay` efektif independen sehingga cukup dicoba satu
per satu di sel pemenang.

RM-c dijalankan dalam dua lapis. **RM-c standar** menguji fusi linear di atas head
juara RM-b (`alpha x k`). **Eksplorasi RM-c** menguji SEMUA head RM-b dengan semua
rumus fusi; karena itu tahap RM-b menyimpan state setiap head di
`checkpoints/rmb_heads/`, bukan hanya juaranya. Juara eksplorasi menggantikan juara
standar hanya bila lolos ambang seri dan bootstrap berpasangan.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [ ]:
import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir

runner = CampaignRunner(out_dir=OUT_DIR)
runner.write_hardware()
print("device   :", runner.device)
print("keluaran :", runner.out_dir)

## 1. Kalibrasi biaya

Jalankan satu konfigurasi RM-a lebih dulu untuk mengukur waktu dan memori
sesungguhnya di mesin ini, sebelum mempertaruhkan berjam-jam pada grid penuh.
Kalau memori kurang, turunkan `MICRO_BATCH` di `.env`; batch efektif tidak
berubah karena selisihnya ditutup akumulasi gradien.

In [ ]:
kalibrasi = runner.run(
    "rma",
    {"lr": 2e-5, "epochs": 5, "batch": 16, "warmup_ratio": 0.1, "weight_decay": 0.01},
    note="kalibrasi biaya: baseline kanonik, sekaligus run #1 grid",
)

per_run = kalibrasi["train_time_s"]
print(f"satu run RM-a: {per_run:.0f} s | peak {kalibrasi['peak_mem_mb']:.0f} MB")
print(f"perkiraan 26 run RM-a: {per_run * 26 / 60:.0f} menit")

## 2. Muat rancangan grid

In [ ]:
GRID_DIR = settings.data_dir.parent / "tuning_grids"

def muat_grid(nama: str) -> list[dict]:
    frame = pd.read_csv(GRID_DIR / nama)
    catatan = frame.pop("catatan") if "catatan" in frame.columns else ""
    return [
        {"config": {k: v for k, v in baris.items() if pd.notna(v)},
         "note": catatan.iloc[i] if hasattr(catatan, "iloc") else ""}
        for i, baris in enumerate(frame.to_dict("records"))
    ]

for berkas in sorted(GRID_DIR.glob("*.csv")):
    print(f"  {berkas.name}: {len(pd.read_csv(berkas))} konfigurasi")

## Melanjutkan kampanye yang terputus

`run_batch` menyaring konfigurasi yang sudah ada di riwayat secara default
(`resume=True`), sehingga sel batch di bawah aman dijalankan ulang apa adanya
setelah kernel mati, listrik padam, atau proses dihentikan. Yang sudah selesai
dilewati, penomoran run berlanjut, dan `best.json` tetap terjaga.

Yang hilang saat terputus hanyalah run yang sedang berjalan saat itu; run yang
sudah selesai ditulis atomik ke `runs_{skenario}.csv` begitu selesai.

Perbandingan memakai konfigurasi LENGKAP setelah nilai default diisi, dan untuk
RM-a `micro_batch` dinormalkan ke nilai efektifnya (`min(batch, micro_batch)`) —
nilai itulah yang menentukan ukuran batch di `DataLoader`.

Sel di bawah memperlihatkan apa yang tersisa sebelum batch dijalankan.

In [ ]:
def sisa(skenario: str, berkas: str) -> None:
    permintaan = muat_grid(berkas)
    tersisa = runner.pending_requests(skenario, permintaan)
    print(f"{berkas:34s} {len(permintaan) - len(tersisa):>3d}/{len(permintaan)} selesai, "
          f"{len(tersisa)} tersisa")

sisa("rma", "RMA_TUNING_GRID.csv")
sisa("rma", "RMA_TUNING_GRID_STAGE2.csv")
for berkas in ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
               "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv"):
    sisa("rmb", berkas)
for berkas in ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv"):
    sisa("rmc", berkas)

## 3. RM-a

Grid tahap 1 menggarap tiga sumbu yang saling terkait. Konfigurasi yang gagal
diisolasi ke `runs_rma_errors.csv` dan tidak menghentikan sisa antrean.

In [ ]:
runner.run_batch("rma", muat_grid("RMA_TUNING_GRID.csv"), batch_id="rma_tahap1_grid")

# Dibaca dari riwayat, bukan dari nilai kembalian run_batch: bila seluruh konfigurasi
# sudah tercatat, run_batch mengembalikan tabel kosong.
riwayat_rma = runner.reporter.runs_frame("rma")
riwayat_rma[riwayat_rma["batch_id"] == "rma_tahap1_grid"].nlargest(10, "val_f1_macro")[
    ["run_id", "lr", "epochs", "batch", "val_f1_macro", "val_f1_judi",
     "train_time_s", "is_tie_with_best"]
]

Baca grid sebagai permukaan, bukan daftar. Heatmap `lr x epochs` per nilai
`batch` di `outputs/tuning/figures/` memperlihatkan apakah learning rate optimal
ikut bergeser saat batch berubah. Pemenang yang duduk di tepi grid adalah sinyal
untuk melebarkan rentang, bukan untuk langsung dikunci.

Selisih di bawah 0,15 pp dihitung seri karena hanya ada satu seed; pada kondisi
seri, pilih konfigurasi yang lebih murah.

In [ ]:
runner.run_batch("rma", muat_grid("RMA_TUNING_GRID_STAGE2.csv"),
                 batch_id="rma_tahap2_coordinate")

riwayat_rma = runner.reporter.runs_frame("rma")
riwayat_rma[riwayat_rma["batch_id"] == "rma_tahap2_coordinate"][
    ["run_id", "warmup_ratio", "weight_decay", "val_f1_macro",
     "delta_vs_best_f1_macro_pp", "is_tie_with_best"]
]

## 4. RM-b

In [ ]:
batch_rmb = []
for berkas in ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
               "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv"):
    batch_id = berkas.replace(".csv", "").lower()
    runner.run_batch("rmb", muat_grid(berkas), batch_id=batch_id)
    batch_rmb.append(batch_id)

riwayat_rmb = runner.reporter.runs_frame("rmb")
riwayat_rmb[riwayat_rmb["batch_id"].isin(batch_rmb)].nlargest(10, "val_f1_macro")[
    ["run_id", "head_arch", "hidden_dim", "lr", "epochs", "dropout",
     "val_f1_macro", "train_time_s", "trainable_params"]
]

Setiap konfigurasi RM-b menyimpan state head-nya di
`outputs/tuning/checkpoints/rmb_heads/run_{id}.pt`. Head itulah yang diuji di
eksplorasi RM-c (bagian 6), jadi head yang dieksplorasi sama persis dengan yang
dilatih di sini.

### Pemulihan checkpoint (clone atau instance baru)

Checkpoint tidak ikut git (`outputs/**/checkpoints/` di-gitignore), sedangkan
`best.json` dan riwayat run ikut. Di instance baru `rmb_heads/`, `rmb_best.pt`,
`rmc_best.pt`, dan `rma_best.pt` hilang, dan menjalankan ulang skenarionya tidak
membuatnya kembali: F1 yang sama dengan juara tidak dipromosikan sehingga tidak
disimpan. Sel di bawah membangunnya dari konfigurasi yang tercatat tanpa menambah
baris riwayat atau mengubah `best.json`: head RM-b dilatih ulang (hitungan detik per
head) dan juara RM-a dilatih ulang sekali.

Di GPU yang sama dengan kampanye asli hasilnya seharusnya identik; pergeseran di atas
0,05 pp dicatat sebagai peringatan. Bila checkpoint sudah ada, sel ini tidak
melakukan apa-apa. Bagian 5 (RM-c standar) dan 6 (eksplorasi) serta
`05_final_benchmark.ipynb` membutuhkan hasilnya.

In [ ]:
dipulihkan = runner.restore_checkpoints(include_rma=True)
print(dipulihkan)

## 5. RM-c standar

RM-c mewarisi head RM-b terbaik, jadi kampanye ini harus dijalankan SETELAH
RM-b selesai. Karena tidak ada training sama sekali, ratusan kombinasi
`alpha x k` selesai dalam hitungan detik.

In [ ]:
batch_rmc = []
for berkas in ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv"):
    batch_id = berkas.replace(".csv", "").lower()
    runner.run_batch("rmc", muat_grid(berkas), batch_id=batch_id)
    batch_rmc.append(batch_id)

riwayat_rmc = runner.reporter.runs_frame("rmc")
riwayat_rmc[riwayat_rmc["batch_id"].isin(batch_rmc)].nlargest(10, "val_f1_macro")[
    ["run_id", "alpha", "k", "weighting", "val_f1_macro", "val_f1_judi", "eval_time_s"]
]

## 6. RM-c eksplorasi: seluruh head RM-b x seluruh rumus fusi

RM-c standar menguji satu head. Bagian ini menguji RAC di atas SETIAP head RM-b
dengan lima rumus fusi (linear produksi, Rumus 1 sampai 4) dan k yang disapu,
lihat `tuning_grids/RMC_EXPLORATION_GRID.md`. Tidak ada pelatihan: head dimuat
dari `checkpoints/rmb_heads/` (dipulihkan di bagian 4 pada instance baru), indeks
FAISS hanya dari train, dan split test tidak dibuka.

Aturan seleksi: F1-macro validation, F1 judi sebagai pemecah seri, selisih di
bawah 0,15 pp dihitung seri sehingga head yang lebih murah dan fusi linear
menang. Karena kandidatnya ribuan, pemenangnya rawan bias seleksi; penantang
baru hanya menggantikan juara standar bila selisihnya melampaui 0,15 pp DAN
lolos bootstrap berpasangan pada split validation.

In [ ]:
from src.services.rmc_exploration import load_exploration_grid

grid_eksplorasi = load_exploration_grid(GRID_DIR / "RMC_EXPLORATION_GRID.csv")
eksplorasi = runner.explore_rmc(grid_eksplorasi)

print(f"{len(grid_eksplorasi)} konfigurasi fusi per head, "
      f"{len(eksplorasi['runs'])} evaluasi seluruhnya")
eksplorasi["per_formula"].round(4)

`shared_*` adalah konfigurasi SERAGAM terbaik tiap rumus, yaitu (alpha, k) dengan
rata-rata F1-macro tertinggi lintas semua head. Angka ini menjawab apakah rumus
membantu tanpa disetel per head, dan lebih jujur daripada `best_*` yang dipilih
dari puluhan kandidat per head.

In [ ]:
per_head = eksplorasi["per_head"]
print(f"RAC membantu di {per_head['rac_helps'].sum()} dari {len(per_head)} head "
      f"(kenaikan di atas ambang seri {settings.tie_threshold_pp} pp)")
print("rumus terbaik per head:", per_head["best_formula"].value_counts().to_dict())

per_head[[
    "rmb_run_id", "head_arch", "hidden_dim", "epochs", "lr", "head_params",
    "val_f1_rmb", "best_formula", "best_alpha", "best_k", "val_f1_rac_best",
    "gain_best_pp", "on_pareto",
]].round(4)

In [ ]:
eksplorasi["runs"].nlargest(10, ["val_f1_macro", "val_f1_judi"])[
    ["rmb_run_id", "formula", "alpha", "k", "weighting", "val_f1_macro",
     "val_f1_judi", "gain_pp"]
].round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, (kiri, kanan) = plt.subplots(1, 2, figsize=(12, 4.5))

kiri.scatter(per_head["val_f1_rmb"], per_head["gain_best_pp"])
kiri.axhline(settings.tie_threshold_pp, color="gray", linestyle="--", label="ambang seri")
kiri.axhline(0, color="black", linewidth=0.8)
kiri.set_xlabel("F1-macro head RM-b sendiri (validation)")
kiri.set_ylabel("kenaikan terbaik dari RAC (pp)")
kiri.set_title("Kenaikan RAC vs kekuatan head")
kiri.legend(fontsize=8)
kiri.grid(alpha=0.3)

linear = eksplorasi["runs"].query("formula == 'linear'")
kurva = linear.groupby(["rmb_run_id", "alpha"])["gain_pp"].max().unstack("alpha")
for _, baris in kurva.iterrows():
    kanan.plot(kurva.columns, baris.values, color="tab:blue", alpha=0.25)
kanan.plot(kurva.columns, kurva.mean(), color="tab:red", linewidth=2.5, label="rata-rata head")
kanan.axhline(0, color="black", linewidth=0.8)
kanan.set_xlabel("alpha (fusi linear)")
kanan.set_ylabel("kenaikan dari RAC (pp), k terbaik per alpha")
kanan.set_title("Kurva alpha per head")
kanan.legend(fontsize=8)
kanan.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(runner.exploration_dir / "rmc_exploration.png", dpi=150)
plt.show()

### Putusan juara RM-c

Penantang adalah konfigurasi terbaik eksplorasi menurut aturan seleksi di atas.
Ia dibandingkan dengan juara standar pada prediksi validation yang sama lewat
bootstrap berpasangan. Bila menang, `best.json` dan `checkpoints/rmc_best.pt`
diganti; `rmc_best.pt` lalu memuat head dan rumus fusi penantang, dan
`05_final_benchmark.ipynb` memuat RM-c dari sana. Bila kalah, juara standar
tetap. Keduanya hasil sah; catatannya di `rmc_exploration/champion_decision.json`.

In [ ]:
keputusan = runner.decide_rmc_champion(eksplorasi["runs"])

print(f"pemenang     : {keputusan['winner']}")
print(f"penantang    : {keputusan['challenger']}")
print(f"petahana     : {keputusan['incumbent']}")
print(f"selisih F1   : {keputusan['delta_pp']:+.3f} pp "
      f"(CI95 {keputusan['ci95_pp'][0]:+.3f} sampai {keputusan['ci95_pp'][1]:+.3f}; "
      f"ambang seri {keputusan['tie_threshold_pp']} pp)")

## 7. Juara tiap skenario

In [ ]:
import json

best = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))
for skenario, entri in best.items():
    print(f"{skenario}: run #{entri['run_id']} | val F1-macro {entri['val_f1_macro']:.4f}")
    print(f"     {entri['config']}\n")

In [ ]:
summary = json.loads((OUT_DIR / "tuning_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Ringkasan

Seluruh angka di atas berasal dari split validation. Split test masih tertutup
dan baru dibuka satu kali di `05_final_benchmark.ipynb`.

Kalau ingin menambah konfigurasi setelah membaca hasil, panggil `runner.run`
atau `runner.run_batch` lagi: riwayat menumpuk dan penomoran run berlanjut.